In [24]:
# =============================================================================
# 資料準備腳本：專門處理 HRDataset_v2，並產出給通用平台測試用的乾淨資料
# 輸出檔案：cleaned_hr_data.csv
# =============================================================================
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
import os

In [25]:
#  PHASE 1｜載入資料
DATA_DIR = os.environ.get("HR_DATA_DIR", "./data/raw/") # 預設讀同目錄，也可用環境變數覆蓋

emp  = pd.read_csv(DATA_DIR + "Employee.csv")
perf = pd.read_csv(DATA_DIR + "PerformanceRating.csv")
edu  = pd.read_csv(DATA_DIR + "EducationLevel.csv")
print(f"[載入] Employee: {emp.shape}, PerformanceRating: {perf.shape}")

[載入] Employee: (1470, 23), PerformanceRating: (6709, 11)


In [26]:
#  PHASE 2｜資料清理
emp.replace([' ', 'NA', 'N/A', 'na'], pd.NA, inplace=True)
emp.columns = emp.columns.str.strip()
emp[emp.select_dtypes(include='object').columns] = \
    emp.select_dtypes(include='object').apply(lambda x: x.str.strip())

# Education 數字代碼 → 文字
emp['Education'] = emp['Education'].replace(
    edu.set_index('EducationLevelID')['EducationLevel'].str.strip().to_dict()
)
emp['HireDate'] = pd.to_datetime(emp['HireDate'], errors='coerce')

# Ethnicity 統一化
emp['Ethnicity'] = emp['Ethnicity'].astype(str).str.strip().replace({
    'Asian or Asian American'        : 'Asian',
    'Mixed or multiple ethnic groups': 'Mixed',
    'Black or African American'      : 'Black',
    'American Indian or Alaska Native': 'Native American',
    'Native Hawaiian'                : 'Pacific Islander',
})

# 邏輯矛盾過濾
before = len(emp)
emp = emp[
    (emp['YearsWithCurrManager']    <= emp['YearsAtCompany']) &
    (emp['YearsSinceLastPromotion'] <= emp['YearsAtCompany']) &
    (emp['YearsAtCompany']          <= emp['Age'])
].copy()
print(f"[清理] 移除邏輯矛盾：{before - len(emp)} 筆，剩餘 {len(emp)} 筆")

[清理] 移除邏輯矛盾：0 筆，剩餘 1470 筆


In [27]:
#  PHASE 3｜薪資極端值處理
def experience_level(x):
    return 'Junior' if x < 2 else ('Mid' if x < 5 else 'Senior')

emp['ExperienceLevel'] = emp['YearsInMostRecentRole'].apply(experience_level)
emp_capped = emp.copy()
outlier_idx = []

for (role, exp), group in emp.groupby(['JobRole', 'ExperienceLevel'], observed=True):
    if len(group) < 4: continue
    Q1, Q3 = group['Salary'].quantile(0.25), group['Salary'].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    emp_capped.loc[group.index, 'Salary_Capped'] = group['Salary'].clip(lower, upper)
    outlier_idx.extend(
        group[(group['Salary'] < lower) | (group['Salary'] > upper)].index.tolist()
    )

emp_capped.drop(index=list(set(outlier_idx)), inplace=True, errors='ignore')
emp_capped['Salary_Capped'] = emp_capped['Salary_Capped'].fillna(emp_capped['Salary'])
print(f"[薪資處理] Capping 後：{len(emp_capped)} 筆")

[薪資處理] Capping 後：1409 筆


In [28]:
#  PHASE 4｜合併績效資料 + 特徵工程
perf['ReviewDate'] = pd.to_datetime(perf['ReviewDate'], format='%m/%d/%Y')
perf['ReviewYear'] = perf['ReviewDate'].dt.year
df = emp_capped.merge(perf.drop(columns=['PerformanceID']), on='EmployeeID', how='inner')
df = df.dropna(subset=['ManagerRating', 'SelfRating']).reset_index(drop=True)
print(f"[合併] {len(df)} 筆，{df['EmployeeID'].nunique()} 名員工")

df['RatingGap']  = df['ManagerRating'] - df['SelfRating']
df['YearWeight'] = df['ReviewYear'] - df['ReviewYear'].min() + 1

def weighted_avg(sub_df, col):
    return (sub_df[col] * sub_df['YearWeight']).sum() / sub_df['YearWeight'].sum()

w_list = []
for eid, g in df.groupby('EmployeeID'):
    w_list.append({
        'EmployeeID': eid,
        'X_WorkLifeBalance'  : weighted_avg(g, 'WorkLifeBalance'),
        'X_EnvironmentSatisfaction'  : weighted_avg(g, 'EnvironmentSatisfaction'),
        'X_JobSatisfaction'  : weighted_avg(g, 'JobSatisfaction'),
        'X_RelationshipSatisfaction'  : weighted_avg(g, 'RelationshipSatisfaction'),
        'X_TrainingOpportunitiesTaken': weighted_avg(g, 'TrainingOpportunitiesTaken'),
    })
weighted_df = pd.DataFrame(w_list)
weighted_df['X_EngagementRate'] = weighted_df[['X_WorkLifeBalance','X_EnvironmentSatisfaction','X_JobSatisfaction','X_RelationshipSatisfaction','X_TrainingOpportunitiesTaken']].mean(axis=1)

df_latest = df.sort_values('ReviewYear').groupby('EmployeeID').last().reset_index()
df_emp    = df_latest.merge(weighted_df, on='EmployeeID', how='left')

def compute_trend(series):
    if len(series) < 2: return 0.0
    years = np.arange(len(series)).reshape(-1, 1)
    return LinearRegression().fit(years, series.values).coef_[0]

t_list = []
for eid, g in df.sort_values('ReviewYear').groupby('EmployeeID'):
    t_list.append({
        'EmployeeID'            : eid,
        'T_JobSatisfaction' : compute_trend(g['JobSatisfaction']),
        'T_WorkLifeBalance' : compute_trend(g['WorkLifeBalance']),
        'T_ManagerRating'   : compute_trend(g['ManagerRating']),
    })
df_emp = df_emp.merge(pd.DataFrame(t_list), on='EmployeeID', how='left')
print(f"[特徵工程] 每人一筆：{len(df_emp)} 筆")

[合併] 6383 筆，1224 名員工
[特徵工程] 每人一筆：1224 筆


In [29]:
#  PHASE 4.5｜SEM 前置：Experience 構面題項信度檢驗 (Cronbach's α)
satisfaction_items = df[['EnvironmentSatisfaction', 'JobSatisfaction',
                         'RelationshipSatisfaction', 'WorkLifeBalance']].astype(float)

def cronbach_alpha(items):
    k = items.shape[1]
    item_vars = items.var(ddof=1).sum()        # 各題變異數總和
    total_var = items.sum(axis=1).var(ddof=1)  # 總分變異數
    return (k / (k - 1)) * (1 - item_vars / total_var)

alpha = cronbach_alpha(satisfaction_items)
print(f"[信度檢驗] Experience 構面 Cronbach's α = {alpha:.4f}（≥0.7 佳，≥0.6 可接受）")

[信度檢驗] Experience 構面 Cronbach's α = 0.2259（≥0.7 佳，≥0.6 可接受）


In [30]:
#  PHASE 5｜SEM 結構方程模型
try:
    from semopy import Model
    sem_data = df[[
        'EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction',
        'WorkLifeBalance', 'TrainingOpportunitiesWithinYear',
        'TrainingOpportunitiesTaken', 'ManagerRating', 'SelfRating', 'RatingGap'
    ]].dropna().astype(float)

    model_desc = """
    Experience =~ EnvironmentSatisfaction + JobSatisfaction + RelationshipSatisfaction + WorkLifeBalance
    TrainingOpportunitiesTaken ~ TrainingOpportunitiesWithinYear
    ManagerRating ~ Experience + TrainingOpportunitiesTaken
    RatingGap ~ Experience + ManagerRating
    """

    # Measurement model: Experience
    # Training mechanism: TrainingOpportunitiesTaken 
    # Performance model: ManagerRating
    # Bias outcome: RatingGap

    sem_model = Model(model_desc)
    sem_model.fit(sem_data)
    
    df_wf = df.loc[sem_data.index].copy()
    df_wf['Experience_SEM'] = sem_model.predict_factors(sem_data)['Experience'].values
    exp_latest = df_wf.sort_values('ReviewYear').groupby('EmployeeID')['Experience_SEM'].last().reset_index()
    df_emp = df_emp.merge(exp_latest, on='EmployeeID', how='left')
    print(f"[SEM] 完成，有效：{df_emp['Experience_SEM'].notna().sum()} 筆")

except ImportError:
    print("[提示] semopy 未安裝，以加權平均代替。pip install semopy 可啟用 SEM")
    df_emp['Experience_SEM'] = df_emp['X_EngagementRate']

[提示] semopy 未安裝，以加權平均代替。pip install semopy 可啟用 SEM


In [31]:
# =============================================================================
# 關鍵新增：將清理與特徵工程完畢的資料，匯出成給通用平台吃的 CSV 檔案
# =============================================================================
df_emp.to_csv(os.path.join(DATA_DIR, 'cleaned_hr_data.csv'), index=False, encoding='utf-8-sig')
print(f"\n🎉 [資料準備完成] 已經成功產出 cleaned_hr_data.csv (共 {len(df_emp)} 筆)！")
print(f"請將此檔案上傳至你的 Streamlit 網頁平台進行偏誤分析。")


🎉 [資料準備完成] 已經成功產出 cleaned_hr_data.csv (共 1224 筆)！
請將此檔案上傳至你的 Streamlit 網頁平台進行偏誤分析。


1. 時間加權滿意度指標 (Time-Weighted Metrics)_X
這些特徵改良了傳統「只看平均」的盲點，採用了時間加權（越近期的數據，權重越重），用來反映員工「當下」的真實狀態。
- X_WorkLifeBalance：工作與生活平衡 (Work-Life Balance) 的加權平均。
- X_EnvironmentSatisfaction：工作環境滿意度 (Environment Satisfaction) 的加權平均。
- X_JobSatisfaction：工作內容滿意度 (Job Satisfaction) 的加權平均。
- X_RelationshipSatisfaction：人際關係滿意度 (Relationship Satisfaction) 的加權平均。
- X_TrainingOpportunitiesTaken：實際參與培訓機會 (Training Opportunities Taken) 的加權平均。
- X_EngagementRate：將上述五項指標再次平均，得出的**「員工整體綜合滿意度」**。

2. 歷史趨勢指標 (Trend Metrics)_T
這組特徵利用線性迴歸（Linear Regression）算出斜率，賦予了模型「看見時間動態」的能力。
- T_JobSatisfaction：工作滿意度的變化趨勢。正值代表「越來越滿意」，負值代表「逐漸失去熱情」。
- T_WorkLifeBalance：工作生活平衡的變化趨勢。
- T_ManagerRating：過去歷年主管評分的變化趨勢。正值代表「績效正在進步」，負值代表「績效正在下滑」。

3. 高階潛在變數 (Latent Variable)
- Experience_SEM：這是透過結構方程模型 (SEM) 提煉出來的「員工體驗分數」。它將多個高度相關的問卷題目（如滿意度、培訓資源）壓縮成一個純淨的單一數值，能有效避免機器學習中的「共線性（Multicollinearity）」問題，是這份資料中含金量最高的客觀特徵之一。「先以 Cronbach's α 檢驗題項信度，確認可靠後再以 SEM 萃取潛在變數」而非「建構效度的前置」，因為 α 量的是信度而非建構效度，後者才是 SEM 因素負荷量／配適指標負責的。